# Notebook 05: Principal Component Analysis (PCA) & Biplot Visualisation
**Project**: Maternal-Child Health Profiles Across Countries Through Unsupervised Clustering

### Objectives:
1. Apply PCA to standardized 17-dimensional maternal-child health data.
2. Evaluate individual and cumulative explained variance.
3. Compute feature loadings and examine variable contributions (biplots).
4. Project observations into 2D space with cluster overlays.

In [ ]:
import sys
import os
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

import config
from src.data_loader import get_latest_cross_sectional_dataset
from src.preprocessing import HealthDataPreprocessor
from src.dimensionality_reduction import HealthPCA

### 1. Fit PCA & Variance Summary

In [ ]:
raw_data = get_latest_cross_sectional_dataset(min_year=2000)
preprocessor = HealthDataPreprocessor(max_missing_country_pct=35.0, imputation_strategy="median")
_, _, standardized = preprocessor.fit_transform_pipeline(raw_data, save_artifacts=False)

X = standardized[config.FEATURE_COLUMNS].values
pca = HealthPCA()
X_pca = pca.fit_transform(X, feature_names=config.FEATURE_COLUMNS)

summary_var = pca.get_variance_summary()
summary_var.head(6)

### 2. Feature Loadings (PC1 and PC2)

In [ ]:
print("Top Contributors to PC1:")
print(pca.get_top_contributors(component=1, top_n=5))

print("\nTop Contributors to PC2:")
print(pca.get_top_contributors(component=2, top_n=5))

### 3. PCA 2D Cluster Projection

In [ ]:
km = KMeans(n_clusters=4, random_state=config.RANDOM_SEED, n_init=config.N_INIT)
labels = km.fit_predict(X)

country_names = standardized["CountryName"].tolist()
proj_df = pca.create_projection_dataframe(X, country_names, cluster_labels=labels)

plt.figure(figsize=(12, 8))
colors = ["#2ca02c", "#1f77b4", "#d62728", "#ff7f0e"]
for c in range(4):
    sub = proj_df[proj_df["Cluster"] == c]
    plt.scatter(sub["PC1"], sub["PC2"], label=f"Cluster {c+1} (N={len(sub)})", color=colors[c], s=80, alpha=0.85)

plt.title(f"PCA 2D Cluster Separation (PC1: {pca.explained_variance_ratio_[0]*100:.1f}%, PC2: {pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.xlabel("PC1 (Maternal Care & Health Infrastructure)")
plt.ylabel("PC2 (Child Nutrition & Immunization)")
plt.axhline(0, color="gray", linestyle=":", alpha=0.5)
plt.axvline(0, color="gray", linestyle=":", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()